# 08 — Baselines clássicos
Dummy, Regressão Logística (L2), PCA + SVM (RBF) e Random Forest. Validação cruzada 5-fold **apenas no treino**; o teste é usado uma única vez ao final.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score, f1_score
import config

d = np.load(config.ARQ_DATASET, allow_pickle=True)
Xtr, ytr, Xte, yte = d["X_train"], d["y_train"], d["X_test"], d["y_test"]
print(Xtr.shape, Xte.shape)

In [ ]:
S = config.SEED
modelos = {
    "Dummy": DummyClassifier(strategy="prior"),
    "LogReg_L2": Pipeline([("sc", StandardScaler()),
                           ("clf", LogisticRegression(C=0.001, max_iter=2000, class_weight="balanced"))]),
    "PCA_SVM_RBF": Pipeline([("sc", StandardScaler()), ("pca", PCA(n_components=100, random_state=S)),
                             ("clf", SVC(C=1.0, kernel="rbf", class_weight="balanced",
                                         probability=True, random_state=S))]),
    "RandomForest": RandomForestClassifier(n_estimators=500, class_weight="balanced_subsample",
                                           n_jobs=-1, random_state=S),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=S)
scoring = {"auc": "roc_auc", "bacc": "balanced_accuracy", "f1": "f1"}

In [ ]:
linhas_cv = []
for nome, m in modelos.items():
    r = cross_validate(m, Xtr, ytr, cv=cv, scoring=scoring, n_jobs=1)
    linhas_cv.append({"modelo": nome,
                      **{f"{k}_media": r[f"test_{k}"].mean() for k in scoring},
                      **{f"{k}_dp": r[f"test_{k}"].std() for k in scoring}})
    print(nome, "ok")
res_cv = pd.DataFrame(linhas_cv).round(3)
display(res_cv)

In [ ]:
linhas_te, probas = [], {}
for nome, m in modelos.items():
    m.fit(Xtr, ytr)
    p = m.predict_proba(Xte)[:, 1]
    pred = (p >= 0.5).astype(int)
    probas[nome] = p
    linhas_te.append({"modelo": nome, "auc": roc_auc_score(yte, p),
                      "bacc": balanced_accuracy_score(yte, pred),
                      "acc": accuracy_score(yte, pred), "f1": f1_score(yte, pred)})
res_te = pd.DataFrame(linhas_te).round(3)
display(res_te)

In [ ]:
res_cv.to_csv(config.RESULTADOS / "baselines_cv.csv", index=False)
res_te.to_csv(config.RESULTADOS / "baselines_teste.csv", index=False)
np.savez(config.RESULTADOS / "proba_baselines.npz", **probas)
print("Resultados salvos em", config.RESULTADOS)